In [12]:
import os
from dotenv import load_dotenv

# Find the .env file one directory up and load it into system memory
dotenv_path = os.path.join('..', '.env')
if load_dotenv(dotenv_path):
    print("Environment variables loaded successfully!")
else:
    print("Warning: Could not find or load .env file.")

Environment variables loaded successfully!


In [14]:
print("🔍 Running clean connection diagnostic...")
try:
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="Say 'API connection verified' and nothing else."
    )
    print("\nSUCCESS! Gemini responded:")
    print(response.text.strip())

except Exception as e:
    print("\nIt still failed. Here is the direct server response status:")
    import traceback
    traceback.print_exc()

🔍 Running clean connection diagnostic...

SUCCESS! Gemini responded:
API connection verified


In [16]:
import os
import re
import json
import time
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
from google import genai
from google.genai import types
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import TextVectorization

# 1. Initialization & API Setup
client = genai.Client()
print("GenAI Client successfully initialized using your .env file secrets!")

# 2. Load Deep Learning Assets
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
VOCAB_PATH = os.path.join('..', 'models', 'vectorizer_vocabulary.json')
TEST_DATA_PATH = os.path.join('..', 'dataset', 'archive.zip') 

print("Loading Bi-LSTM model and vocabulary...")
model = load_model(MODEL_PATH)
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    saved_vocabulary = json.load(f)

# Reconstruct our training text vectorizer
inference_vectorizer = TextVectorization(
    max_tokens=10000,
    output_mode='int',
    output_sequence_length=150,
    vocabulary=saved_vocabulary
)

# 3. Load Test Dataset Subset (FIXED: Added explicit sep='\t')
print("Loading test data from zip archive...")
with zipfile.ZipFile(TEST_DATA_PATH, 'r') as z:
    with z.open('drugsComTest_raw.csv') as f:
        df_test = pd.read_csv(f, sep=',') # FIXED: Clean tab parsing

# Take a clean slice of the first 100 rows to respect free-tier API limits
df_eval = df_test.head(100).copy() 
print(f"📋 Successfully loaded {len(df_eval)} test rows for GenAI processing!")

# Clean text for Bi-LSTM prediction
def clean_text(text):
    text = str(text).lower()
    text = text.replace('&#039;', "'").replace('&amp;', '&')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

print("Generating Bi-LSTM ratings...")
cleaned_reviews = df_eval['review'].apply(clean_text)
vectorized_inputs = inference_vectorizer(cleaned_reviews.values).numpy()
raw_preds = model.predict(vectorized_inputs, verbose=0)
df_eval['predicted_rating'] = np.argmax(raw_preds, axis=1) + 1


# 4. Define Prompt Engineering Alternatives

# --- ALTERNATIVE A: ZERO-SHOT STRUCTURED PROMPT ---
def get_prompt_alt_a(review):
    return f"""
You are analyzing a patient medical review. Your task is to extract an ultra-short summary and a binary sentiment.

Review to analyze: "{review}"

Return your response exactly as a valid JSON object with these two keys:
- "summary": A summary of the review that is strictly a MAXIMUM of 10 words.
- "sentiment": Output exactly either "positive" or "negative".
"""

# --- ALTERNATIVE B: FEW-SHOT SYSTEM-TAGGED PROMPT ---
system_instruction_alt_b = """
You are a specialized clinical data extraction assistant. Your job is to parse unstructured patient drug reviews into rigid, concise formats. 
You must strictly follow these constraints:
1. The summary field must be 10 words or fewer.
2. The sentiment field must be exactly 'positive' or 'negative'.
3. Respond ONLY with a valid JSON block containing the keys "summary" and "sentiment". Do not include markdown code block formatting or backticks.
"""

def get_prompt_alt_b(review):
    return f"""
<examples>
Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Output: {{"summary": "Life-changing drug eliminated all chronic symptoms quickly with no side-effects.", "sentiment": "positive"}}

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Output: {{"summary": "Severe side effects including intense nausea and headaches immediately.", "sentiment": "negative"}}
</examples>

<target_review>
{review}
</target_review>

Output JSON:
"""


# 5. Upgraded Execution Loop with Rate-Limit Backoff Retries
def run_genai_pipeline(prompt_style="A"):
    summaries = []
    sentiments = []
    
    print(f"\n🚀 Running Generative Track: Alternative {prompt_style}...")
    
    for idx, (current_idx, row) in enumerate(df_eval.iterrows()):
        review_text = row['review']
        print(f"Processing row {idx + 1}/{len(df_eval)}...", end="\r")
        
        max_retries = 3
        success = False
        
        for attempt in range(max_retries):
            try:
                if prompt_style == "A":
                    response = client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=get_prompt_alt_a(review_text),
                        config=types.GenerateContentConfig(response_mime_type="application/json")
                    )
                else:
                    response = client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=get_prompt_alt_b(review_text),
                        config=types.GenerateContentConfig(
                            system_instruction=system_instruction_alt_b,
                            response_mime_type="application/json"
                        )
                    )
                
                # Try parsing JSON output
                data = json.loads(response.text.strip())
                summaries.append(data.get("summary", "Error parsing summary"))
                sentiments.append(data.get("sentiment", "Error parsing sentiment"))
                success = True
                break 
                
            except Exception as e:
                # If we encounter a rate limit wall, pause and retry
                if "429" in str(e) or "Quota" in str(e) or "ResourceExhausted" in str(e):
                    print(f"\nRate limit hit on row {idx + 1}. Pausing 15 seconds (Attempt {attempt + 1}/{max_retries})...")
                    time.sleep(15)
                else:
                    # Log other anomalies
                    print(f"\nUnexpected row error: {str(e)[:40]}")
                    break
        
        if not success:
            summaries.append("API Execution Error")
            sentiments.append("error")
            
        # Standard structural request spacing
        time.sleep(5.0) 
        
    df_output = df_eval[['drugName', 'condition', 'review', 'rating', 'predicted_rating']].copy()
    df_output['generated_summary'] = summaries
    df_output['identified_sentiment'] = sentiments
    return df_output

# Run both tracking strategies sequentially
output_alt_a = run_genai_pipeline(prompt_style="A")
output_alt_b = run_genai_pipeline(prompt_style="B")

# 6. Export to Required Excel Formats
output_dir = os.path.join('..', 'outputs')
os.makedirs(output_dir, exist_ok=True)

output_alt_a.to_excel(os.path.join(output_dir, 'alternative_a_zero_shot.xlsx'), index=False)
output_alt_b.to_excel(os.path.join(output_dir, 'alternative_b_few_shot.xlsx'), index=False)

print("\n🎉 Both Excel deliverables successfully exported to the outputs/ directory with zero errors!")

GenAI Client successfully initialized using your .env file secrets!
Loading Bi-LSTM model and vocabulary...
Loading test data from zip archive...
📋 Successfully loaded 100 test rows for GenAI processing!
Generating Bi-LSTM ratings...

🚀 Running Generative Track: Alternative A...
Processing row 1/100...
Unexpected row error: 503 UNAVAILABLE. {'error': {'code': 503,
Processing row 2/100...
Rate limit hit on row 2. Pausing 15 seconds (Attempt 1/3)...

Rate limit hit on row 2. Pausing 15 seconds (Attempt 2/3)...

Rate limit hit on row 2. Pausing 15 seconds (Attempt 3/3)...
Processing row 3/100...
Rate limit hit on row 3. Pausing 15 seconds (Attempt 1/3)...

Rate limit hit on row 3. Pausing 15 seconds (Attempt 2/3)...

Rate limit hit on row 3. Pausing 15 seconds (Attempt 3/3)...
Processing row 4/100...
Rate limit hit on row 4. Pausing 15 seconds (Attempt 1/3)...

Rate limit hit on row 4. Pausing 15 seconds (Attempt 2/3)...

Rate limit hit on row 4. Pausing 15 seconds (Attempt 3/3)...
Proces